# 02 Training DenseNet
This notebook trains DenseNet121 with reproducible splits, better checkpointing, and richer training diagnostics.


In [ ]:
import csv
import json
import random
import time
from contextlib import nullcontext
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, models, transforms
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

# Configuration (edit here)
DATA_DIR = Path("data")
MODELS_DIR = Path("models")
SPLIT_FILE = MODELS_DIR / "split_indices.json"
CLASS_NAMES_FILE = MODELS_DIR / "class_names.txt"

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0
SEED = 42

NUM_EPOCHS = 20
EARLY_STOP_PATIENCE = 5
BASE_LR = 1e-4
FINE_TUNE_LR = 5e-5
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05

FREEZE_FEATURES_AT_START = True
FREEZE_EPOCHS = 2  # unfreeze after this many epochs

RESUME_PATH = MODELS_DIR / "densenetAllLatest.pth"
BEST_PATH = MODELS_DIR / "densenetAllBest.pth"
LATEST_PATH = MODELS_DIR / "densenetAllLatest.pth"
HISTORY_CSV = MODELS_DIR / "training_history.csv"
CURVE_PNG = MODELS_DIR / "training_curves.png"
TEST_METRICS_TXT = MODELS_DIR / "test_metrics.txt"


In [ ]:
# Device + reproducibility
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(SEED)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Using device: {device}")


In [ ]:
class TransformSubset(Dataset):
    def __init__(self, base_dataset, indices, transform):
        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        sample_idx = self.indices[idx]
        path, target = self.base_dataset.samples[sample_idx]
        image = self.base_dataset.loader(path)
        if self.transform is not None:
            image = self.transform(image)
        return image, target

def stratified_split_indices(targets, train_ratio, val_ratio, seed):
    per_class_indices = defaultdict(list)
    for idx, label in enumerate(targets):
        per_class_indices[label].append(idx)

    rng = random.Random(seed)
    train_indices, val_indices, test_indices = [], [], []
    for _, cls_indices in per_class_indices.items():
        rng.shuffle(cls_indices)
        n_total = len(cls_indices)
        n_train = int(n_total * train_ratio)
        n_val = int(n_total * val_ratio)

        train_indices.extend(cls_indices[:n_train])
        val_indices.extend(cls_indices[n_train:n_train + n_val])
        test_indices.extend(cls_indices[n_train + n_val:])

    rng.shuffle(train_indices)
    rng.shuffle(val_indices)
    rng.shuffle(test_indices)
    return train_indices, val_indices, test_indices

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


In [ ]:
# Build/load splits
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Data directory not found: {DATA_DIR.resolve()}")

base_dataset = datasets.ImageFolder(root=str(DATA_DIR))
targets = [label for _, label in base_dataset.samples]
class_names = base_dataset.classes
num_classes = len(class_names)

if SPLIT_FILE.exists():
    split_data = json.loads(SPLIT_FILE.read_text(encoding="utf-8"))
    train_indices = split_data["train_indices"]
    val_indices = split_data["val_indices"]
    test_indices = split_data["test_indices"]
    print(f"Loaded split file: {SPLIT_FILE}")
else:
    train_indices, val_indices, test_indices = stratified_split_indices(targets, 0.70, 0.15, SEED)
    split_payload = {
        "seed": SEED,
        "ratios": {"train": 0.70, "val": 0.15, "test": 0.15},
        "class_to_idx": base_dataset.class_to_idx,
        "train_indices": train_indices,
        "val_indices": val_indices,
        "test_indices": test_indices,
    }
    SPLIT_FILE.write_text(json.dumps(split_payload, indent=2), encoding="utf-8")
    print(f"Created split file: {SPLIT_FILE}")

train_dataset = TransformSubset(base_dataset, train_indices, train_transform)
val_dataset = TransformSubset(base_dataset, val_indices, eval_transform)
test_dataset = TransformSubset(base_dataset, test_indices, eval_transform)

pin_memory = device.type == "cuda"
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=pin_memory)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin_memory)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin_memory)

CLASS_NAMES_FILE.write_text("\n".join(class_names) + "\n", encoding="utf-8")

print(f"Classes ({num_classes}): {class_names}")
print(f"Split sizes -> train: {len(train_dataset)}, val: {len(val_dataset)}, test: {len(test_dataset)}")


In [ ]:
# Model setup
try:
    weights = models.DenseNet121_Weights.IMAGENET1K_V1
    model = models.densenet121(weights=weights)
except AttributeError:
    model = models.densenet121(pretrained=True)

in_features = model.classifier.in_features
model.classifier = nn.Linear(in_features, num_classes)

if FREEZE_FEATURES_AT_START and FREEZE_EPOCHS > 0:
    for param in model.features.parameters():
        param.requires_grad = False
    print(f"Feature extractor frozen for first {FREEZE_EPOCHS} epochs")
else:
    print("Training all layers from epoch 1")

model = model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

def make_optimizer(lr):
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    return optim.AdamW(trainable_params, lr=lr, weight_decay=WEIGHT_DECAY)

optimizer = make_optimizer(BASE_LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
)

use_amp = device.type == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

start_epoch = 0
best_val_loss = float("inf")
history = {
    "epoch": [],
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    "lr": [],
}


In [ ]:
# Optional resume
if RESUME_PATH.exists():
    checkpoint = torch.load(RESUME_PATH, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    if "scheduler_state_dict" in checkpoint:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    if use_amp and checkpoint.get("scaler_state_dict") is not None:
        scaler.load_state_dict(checkpoint["scaler_state_dict"])
    start_epoch = checkpoint.get("epoch", 0)
    best_val_loss = checkpoint.get("best_val_loss", best_val_loss)
    old_history = checkpoint.get("history")
    if isinstance(old_history, dict):
        history = old_history
    print(f"Resumed from {RESUME_PATH} at epoch {start_epoch}")
else:
    print("No resume checkpoint found, starting fresh")


In [ ]:
def run_one_epoch(loader, train_mode=True):
    if train_mode:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    pbar = tqdm(loader, desc="train" if train_mode else "val", leave=False)
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        if train_mode:
            optimizer.zero_grad(set_to_none=True)

        amp_ctx = torch.autocast(device_type="cuda", enabled=True) if use_amp else nullcontext()
        with amp_ctx:
            outputs = model(images)
            loss = criterion(outputs, labels)

        if train_mode:
            if use_amp:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()

        preds = outputs.argmax(dim=1)
        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (preds == labels).sum().item()
        total_samples += batch_size

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = total_loss / max(1, total_samples)
    avg_acc = total_correct / max(1, total_samples)
    return avg_loss, avg_acc

def save_history_artifacts(history_dict):
    # Save CSV
    with open(HISTORY_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "train_loss", "train_acc", "val_loss", "val_acc", "lr"])
        rows = zip(
            history_dict["epoch"],
            history_dict["train_loss"],
            history_dict["train_acc"],
            history_dict["val_loss"],
            history_dict["val_acc"],
            history_dict["lr"],
        )
        for row in rows:
            writer.writerow(row)

    # Save curves
    epochs = history_dict["epoch"]
    if not epochs:
        return

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, history_dict["train_loss"], label="train")
    axes[0].plot(epochs, history_dict["val_loss"], label="val")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(epochs, history_dict["train_acc"], label="train")
    axes[1].plot(epochs, history_dict["val_acc"], label="val")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(CURVE_PNG, dpi=250)
    plt.show()
    plt.close(fig)


In [ ]:
# Training loop
epochs_without_improvement = 0
features_unfrozen = not (FREEZE_FEATURES_AT_START and FREEZE_EPOCHS > 0)

for epoch in range(start_epoch, NUM_EPOCHS):
    epoch_start = time.time()

    # Unfreeze feature extractor after warm-up
    if (not features_unfrozen) and (epoch >= FREEZE_EPOCHS):
        for param in model.features.parameters():
            param.requires_grad = True
        optimizer = make_optimizer(FINE_TUNE_LR)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=2,
            min_lr=1e-6,
        )
        features_unfrozen = True
        print(f"Unfroze feature extractor at epoch {epoch + 1}")

    train_loss, train_acc = run_one_epoch(train_loader, train_mode=True)
    val_loss, val_acc = run_one_epoch(val_loader, train_mode=False)

    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]["lr"]

    history["epoch"].append(epoch + 1)
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["lr"].append(current_lr)

    ckpt = {
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict() if use_amp else None,
        "best_val_loss": best_val_loss,
        "val_loss": val_loss,
        "class_names": class_names,
        "history": history,
    }

    torch.save(ckpt, LATEST_PATH)

    improved = val_loss < best_val_loss
    if improved:
        best_val_loss = val_loss
        ckpt["best_val_loss"] = best_val_loss
        torch.save(ckpt, BEST_PATH)
        epochs_without_improvement = 0
        status = "best"
    else:
        epochs_without_improvement += 1
        status = "-"

    epoch_time = time.time() - epoch_start
    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
        f"train_loss {train_loss:.4f} train_acc {train_acc:.4f} | "
        f"val_loss {val_loss:.4f} val_acc {val_acc:.4f} | "
        f"lr {current_lr:.2e} | {status} | {epoch_time:.1f}s"
    )

    save_history_artifacts(history)

    if epochs_without_improvement >= EARLY_STOP_PATIENCE:
        print(f"Early stopping triggered (patience={EARLY_STOP_PATIENCE})")
        break

print(f"Best checkpoint saved at: {BEST_PATH}")
print(f"Latest checkpoint saved at: {LATEST_PATH}")


In [ ]:
# Final test evaluation with best checkpoint
if not BEST_PATH.exists():
    raise FileNotFoundError(f"Best checkpoint not found: {BEST_PATH}")

best_ckpt = torch.load(BEST_PATH, map_location=device)
model.load_state_dict(best_ckpt["model_state_dict"])
model.eval()

test_loss = 0.0
test_correct = 0
test_total = 0

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="test", leave=False):
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        preds = outputs.argmax(dim=1)

        batch_size = labels.size(0)
        test_loss += loss.item() * batch_size
        test_correct += (preds == labels).sum().item()
        test_total += batch_size

avg_test_loss = test_loss / max(1, test_total)
test_acc = test_correct / max(1, test_total)

test_summary = [
    f"best_epoch: {best_ckpt.get('epoch', 'unknown')}",
    f"best_val_loss: {best_ckpt.get('best_val_loss', 'unknown')}",
    f"test_loss: {avg_test_loss:.6f}",
    f"test_accuracy: {test_acc:.6f}",
]
TEST_METRICS_TXT.write_text("\n".join(test_summary) + "\n", encoding="utf-8")

print(f"Test loss: {avg_test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")
print(f"Saved test metrics -> {TEST_METRICS_TXT}")
